# Dual-Pathway Pruning (Using OpenAI and BigQuery)


## Install piglets


If running from within the piglets repository:
```bash
    uv sync --extra examples --extra openai --extra bigquery
``` 
followed by the import below:


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


If running from outside the piglets repository:
```bash
    uv add piglets[openai,bigquery,examples]
``` 


## Set Your Model, Database Schema and Natural Language Question


In [ ]:
import os

MODEL_NAME = "gpt-5.2"
DATABASE_TYPE = "bigquery"
BQ_DATA_PROJECT = "bigquery-public-data"
BQ_DATASET = "stackoverflow"
BQ_BILLING_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT_ID")
QUESTION =  """
    Which tags saw the largest increase in average answer score from 2022 to 2023, 
    considering only questions with at least 5 answers?
"""


## Step 1: Create a Logical Plan


We first have our model verbalise it's hypotheses. This step is schema agnostic and is therefore completed without any knowledge of the actual database. This allows the model to reason about possible approaches without the influence of naming conventions and ambiguous abbreviations in the database schema.

The `piglets` logical planner creates `N` logical plans before aggregating them into a single plan. This is to ensure less potential hypotheses are missed. The approach here errs on the side of maximising recall, therefore we would rather have too many hypotheses to validate than include one that is valuable. In the `LogicalPlanner` the number of logcial plans we create before aggregation is determined by `num_samples`, in this case we have a sample size of 3.


In [ ]:
from piglets import LogicalPlanner, Question


question = Question(natural_language_question=QUESTION)
logical_planner = LogicalPlanner(MODEL_NAME, num_samples=3)
logical_plan = logical_planner.plan(
    question=question,
)


Below we can inspect the sample plans that were first created and the aggregate plan that was created form them:


In [ ]:
print("------------------------------------")
print("Natural Language Question:")
print(logical_plan.natural_language_query)
print("------------------------------------")
print("Sample Plan 1:")
for i, step in enumerate(logical_plan.sample_plans[0].logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")
print("Sample Plan 2:")
for i, step in enumerate(logical_plan.sample_plans[1].logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")
print("Sample Plan 3:")
for i, step in enumerate(logical_plan.sample_plans[2].logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")
print("\nAggregated Logical Plan:")
for i, step in enumerate(logical_plan.logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")


## Step 2: Connect to BigQuery and retrieve the Database Schema


Next we connect to BigQuery. The data lives in the public Stack Overflow dataset at `bigquery-public-data.stackoverflow`.
You still need authenticated BigQuery access. If `GOOGLE_CLOUD_PROJECT` or `GOOGLE_CLOUD_PROJECT_ID` is set, Piglets uses that as the billing/query project while reading from the public data project.
If neither environment variable is set, the notebook does not fail fast, but BigQuery Application Default Credentials may still require a quota or billing project depending on your local auth setup.
Once connected we can retrieve information about our database, which will be useful context for future LLM calls.


In [ ]:
from piglets import BigQueryURL, DatabaseConnector, SearchSpace

database_connector = DatabaseConnector(
    connection=BigQueryURL(
        project_id=BQ_DATA_PROJECT,
        dataset=BQ_DATASET,
        billing_project_id=BQ_BILLING_PROJECT,
    ),
)
search_space = database_connector.add_to_search_space(SearchSpace())


In [ ]:
print("------------------------------------")
print("Database Name: ")
print(search_space.database_schema.name)
print("------------------------------------")
print("Database Tables: ")
for table_schema in search_space.database_schema.table_schemas:
    print("------------------------------------")
    print(f"Table: {table_schema.name}")
    print("Columns:")
    for column_schema in table_schema.column_schemas:
        print(f"  - {column_schema.name} ({column_schema.data_type})")
print("------------------------------------")


## Step 3: Use Dual-Pathway Pruning to Reduce Potemtial Schema


Above we have retrieved a large schema, much of which won't be relevant to our natural language question.
The `piglets` `DualPathwayPruner` uses dual-pathway pruning to reduce the size of this schema.
Dual-pathway pruning first produces a `PreservationSet` and a `DeletionSet`, these are made up of potentially useful tables and columns and obviously useless tables and columns respectively. 
We then take the union of all columns either not present in the `DeletionSet` or are present in the `PreservationSet`. More formally:
$$
D_{\text{pruned}} = (D \setminus C_{\text{del}}) \cup C_{\text{keep}}
$$
for the database schema $D$, a preservation set $C_{\text{keep}}$ and a deletion set $C_{\text{del}}$.


In [ ]:
from piglets import DualPathwayPruner, SearchSpace

dual_pathway_pruner = DualPathwayPruner(model_name=MODEL_NAME)
pruned_search_space = dual_pathway_pruner.dual_pathway_pruning(
    question=question,
    search_space=search_space,
    logical_plan=logical_plan,
)


In [ ]:
print("------------------------------------")
print("Database Name: ")
print(pruned_search_space.database_schema.name)
print("------------------------------------")
print("Database Tables: ")
for table_schema in pruned_search_space.database_schema.table_schemas:
    print("------------------------------------")
    print(f"Table: {table_schema.name}")
    print("Columns:")
    for column_schema in table_schema.column_schemas:
        print(f"  - {column_schema.name} ({column_schema.data_type})")
print("------------------------------------")


You should now see a reduced version of the original schema above.
